# Airbnb calendar Bronze ingestion (F06.2 / F06.3)

Ingests the Toronto Inside Airbnb `calendar.csv.gz` into its own Bronze
data product (`airbnb_calendar_bronze`) - separate from listings and
reviews - reading the immutable F04 manifest landing object.

Since the pre-F07 cleanup (T03) the complete ingestion flow lives ONCE
in `src/airbnb/bronze_flow.py` and is shared by every per-dataset
Bronze notebook: manifest-record resolution by explicit
`source_file_id` (the shared family/entity/version identity is
ambiguous across the three Toronto files, verified in F06.1),
immutable-landing verification (hash + size, full gzip decompression,
header fingerprint, contract-required columns), string-preserving
reads, contract-driven row-quality retention (invalid rows are RETAINED
with `_quality_status` / `_quality_reason`, never dropped), the
unified ingestion manifest (F07.4) and the atomic
`replaceWhere` slice write with reconciliation asserts.

This notebook supplies only the calendar dataset spec, the run widgets
and the result display. City, snapshot date and source family are
derived from the landed manifest record itself - never hard-coded.


In [0]:
# Run-mode controls (F02.2 / F05.3 conventions, applied to the calendar
# dataset in F06.2-T02).
#
# run_mode defaults to "test": every table this notebook touches is
# resolved into the dedicated trippilot_test catalog through
# src.common.environment. "live" resolves the development tables and is
# used only after the test-mode evidence gate has passed (and, for new
# datasets, the licence confirmation).
#
# force_reload is the explicit reprocess authorization: it permits an
# exact rerun of an already-ingested file and the controlled replacement
# of a corrected source file (a different source_file_id).
#
# source_file_id pins the EXACT landed manifest record this run reads:
# the family/entity/version identity (insideairbnb/toronto/2026-09-16 +
# csv.gz) is shared by the listings, calendar and reviews files, so
# resolving by identity alone is ambiguous - the explicit id is
# required (verified in F06.1).
dbutils.widgets.dropdown("run_mode", "test", ["test", "live"])
dbutils.widgets.text("force_reload", "false")
dbutils.widgets.text(
    "source_file_id",
    "09a7ef20b005bc7fb5ee0abdb6ba27f07a2d2e72a62885351442702ef38fefb2",
)

RUN_MODE = dbutils.widgets.get("run_mode")
FORCE_RELOAD = dbutils.widgets.get("force_reload")
SOURCE_FILE_ID = dbutils.widgets.get("source_file_id")

print(f"Run mode: {RUN_MODE} | force_reload: {FORCE_RELOAD}")
print(f"source_file_id: {SOURCE_FILE_ID}")

In [0]:
# Locate the repository root, then run the shared flow (pre-F07 T03).
from pathlib import Path
import sys

start = Path.cwd().resolve()
ROOT = next(
    (
        p
        for p in [start, *start.parents]
        if (p / "src").is_dir() and (p / "config").is_dir()
    ),
    None,
)
if ROOT is None:
    raise RuntimeError(
        "Cannot locate the TripPilot repository root (src/ and config/)."
    )
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.airbnb.bronze_flow import BronzeDatasetSpec, run_bronze_ingestion

# Dataset identity for the calendar path (F06.1 evidence): the shared
# flow derives city/snapshot/source from the landed manifest record.
CALENDAR_SPEC = BronzeDatasetSpec(
    dataset="calendar",
    schema_label="calendar",
    contract_path="contracts/airbnb/bronze_calendar.yml",
    rules_key="bronze_calendar",
    bronze_key="calendar_bronze",
    default_source_file_id="09a7ef20b005bc7fb5ee0abdb6ba27f07a2d2e72a62885351442702ef38fefb2",
)

summary = run_bronze_ingestion(
    CALENDAR_SPEC,
    spark,
    source_file_id=SOURCE_FILE_ID,
    run_mode=RUN_MODE,
    force_reload=FORCE_RELOAD,
)

In [0]:
# Result display: F06.2/F06.3 acceptance evidence (raw sample, retained
# invalid rows, per-identity verification, per-slice summary).
display(summary["raw_df"].limit(5))

if summary["invalid_rows"]:
    from pyspark.sql import functions as F

    display(
        summary["bronze_df"].filter(F.col("_quality_status") == "invalid").limit(20)
    )

display(summary["verification_df"])
display(summary["snapshot_summary"])